In [ ]:
# Instalar dependências necessárias (se ainda não estiverem instaladas)
!pip install ultralytics wandb opencv-python pyyaml

# Importação das bibliotecas
import os
import numpy as np
import pandas as pd
import yaml
import zipfile
import matplotlib.pyplot as plt
import cv2
from IPython.display import HTML, display
from matplotlib import animation
from tqdm import tqdm
from PIL import Image
from ultralytics import YOLO

In [ ]:
# Definição dos caminhos do dataset
data_yaml = {
    "train": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/train/images",
    "val": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/val/images",
    "test": "/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/dataset/test/images",
    "nc": 9,
    "names": [
        "Trafic Light Signal", "Stop Signal", "Speedlimit Signal", "Crosswalk Signal",
        "Crosswalk", "Pedestrian", "Bus", "Car", "Truck"
    ]
}

# Salvar dataset.yaml
with open("dataset.yaml", "w") as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)

# Função para criar um arquivo ZIP do dataset
def create_zip(source_folder, destination_zip):
    """ Compacta um diretório em um arquivo ZIP. """
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zip_ref:
        for root, _, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                zip_ref.write(file_path, arcname=os.path.relpath(file_path, source_folder))

# Função para extrair arquivos ZIP
def extract_zip(zip_file, destination_folder):
    """ Extrai um arquivo ZIP para um diretório específico. """
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(destination_folder)


In [ ]:
# Treinamento do modelo YOLO
model = YOLO("yolo11m.pt")

# Iniciar o treinamento
!yolo task=detect mode=train model=yolo11m.pt data=dataset.yaml epochs=25 imgsz=640

# Criar um ZIP dos resultados
!zip -r runs.zip runs/
display(FileLink("runs.zip"))

# Função para exibir imagens do treinamento
def display_training_results(directory, file_extension=('.jpg', '.png'), images_per_row=2, image_height=10):
    """ Exibe as imagens do treinamento. """
    image_paths = sorted([
        os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(file_extension)
    ])

    if not image_paths:
        print(f"⚠️ Nenhuma imagem encontrada em {directory}. Verifique se o treinamento foi concluído.")
        return

    num_images = len(image_paths)
    num_rows = max(1, (num_images + images_per_row - 1) // images_per_row)
    figsize = (images_per_row * image_height, num_rows * image_height)

    fig, axes = plt.subplots(num_rows, images_per_row, figsize=figsize)
    axes = axes.flatten()

    for i, path in enumerate(image_paths):
        image = Image.open(path)
        axes[i].imshow(image)
        axes[i].axis('off')

    for j in range(len(image_paths), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Exibir resultados do treinamento
display_training_results("runs/detect/train")

# Caminho do melhor modelo treinado
best_path = "runs/detect/train5/weights/best.pt"
source = "dataset/test/images"

# Coletar imagens do dataset de teste
image_paths = sorted([
    os.path.join(dirname, filename)
    for dirname, _, filenames in os.walk(source)
    for filename in filenames if filename.endswith(('.jpg', '.png'))
])

# Carregar o modelo treinado
model = YOLO(best_path)

# Fazer previsões
results = model.predict(source, conf=0.1)

# Criar um DataFrame para armazenar resultados
df = pd.DataFrame(columns=["x", "y", "x2", "y2", "confidence", "class", "file", "i"])

for i, result in enumerate(results):
    arri = pd.DataFrame(result.boxes.data.cpu().numpy()).astype(float)
    path = image_paths[i]
    file = os.path.basename(path)
    arri["file"] = file
    arri["i"] = i
    df = pd.concat([df, arri], ignore_index=True)

df.columns = ["x", "y", "x2", "y2", "confidence", "class", "file", "i"]

# Exibir os resultados
display(df)

# Função para desenhar as caixas delimitadoras
def draw_box(image_index):
    """ Desenha caixas delimitadoras nas imagens com base nos resultados do YOLO. """
    image_path = image_paths[image_index]
    image = cv2.imread(image_path)
    height, width = image.shape[:2]
    filename = os.path.basename(image_path)

    if not df[df["file"] == filename].empty:
        boxes = df[df["file"] == filename].reset_index(drop=True)

        for _, row in boxes.iterrows():
            label = int(row["class"])
            x, y, x2, y2 = map(int, [row["x"], row["y"], row["x2"], row["y2"]])

            cv2.putText(image, f"{label}", (x, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
            cv2.rectangle(image, (x, y), (x2, y2), (0, 255, 0), 2)

    return image

# Criar animação dos resultados
def create_animation(images):
    """ Cria uma animação com as imagens anotadas. """
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis("off")
    image_display = ax.imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))

    def animate_func(frame_index):
        image_display.set_array(cv2.cvtColor(images[frame_index], cv2.COLOR_BGR2RGB))
        return [image_display]

    return animation.FuncAnimation(fig, animate_func, frames=len(images), interval=1000)

# Gerar imagens anotadas
annotated_images = [draw_box(i) for i in tqdm(range(len(image_paths)))]

# Criar e exibir a animação
anim = create_animation(annotated_images)
HTML(anim.to_jshtml())